In [5]:
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────────
INPUT_FILE  = "Datasets/Moral Machine Data/SharedResponses.csv"
OUTPUT_FILE = "SharedResponses.parquet"
CHUNK_SIZE  = 500_000

# ── Dtype definitions ───────────────────────────────────────────────────────
nullable_uint8  = pd.UInt8Dtype()
nullable_int8   = pd.Int8Dtype()
nullable_uint64 = pd.UInt64Dtype()

cols_to_drop = [
    "ExtendedSessionID", "LeftHand", "ScenarioType",
    "Template", "DescriptionShown"
]

nullable_uint8_cols = [
    "ScenarioOrder", "DefaultChoiceIsOmission", "NumberOfCharacters"
]

char_cols = [
    "Man", "Woman", "Pregnant", "Stroller", "OldMan", "OldWoman",
    "Boy", "Girl", "Homeless", "LargeWoman", "LargeMan", "Criminal",
    "MaleExecutive", "FemaleExecutive", "FemaleAthlete", "MaleAthlete",
    "FemaleDoctor", "MaleDoctor", "Dog", "Cat"
]

category_cols_with_nulls    = ["UserCountry3", "DefaultChoice", "NonDefaultChoice"]
category_cols_without_nulls = ["ResponseID", "AttributeLevel", "ScenarioTypeStrict"]
uint8_cols_no_nulls         = ["Intervention", "PedPed", "Barrier", "CrossingSignal"]

# ── Cast function ───────────────────────────────────────────────────────────
def cast_dtypes(chunk):
    chunk = chunk.drop(columns=cols_to_drop, errors="ignore")

    chunk["UserID"] = chunk["UserID"].astype(nullable_uint64)

    for col in nullable_uint8_cols:
        chunk[col] = chunk[col].astype(nullable_uint8)

    chunk["DiffNumberOFCharacters"] = chunk["DiffNumberOFCharacters"].astype(nullable_int8)

    chunk["Man"] = pd.to_numeric(chunk["Man"], errors="coerce").astype(nullable_uint8)
    for col in [c for c in char_cols if c != "Man"]:
        chunk[col] = chunk[col].astype(nullable_uint8)

    for col in category_cols_with_nulls + category_cols_without_nulls:
        chunk[col] = chunk[col].astype("category")

    for col in uint8_cols_no_nulls:
        chunk[col] = chunk[col].astype("uint8")

    return chunk

# ── Read → cast → write ─────────────────────────────────────────────────────
import pyarrow as pa
import pyarrow.parquet as pq

writer = None
for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    chunk = cast_dtypes(chunk)
    table = pa.Table.from_pandas(chunk)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
    writer.write_table(table)
    if i % 10 == 0:
        print(f"Processed {(i + 1) * CHUNK_SIZE:,} rows...")

writer.close()
print(f"\nSaved to {OUTPUT_FILE}")

# ── Verify ──────────────────────────────────────────────────────────────────
df = pd.read_parquet(OUTPUT_FILE)
print("\nDtypes:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Null counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Processed 500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_64547/2956955587.py:61: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 5,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_64547/2956955587.py:61: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 10,500,000 rows...
Processed 15,500,000 rows...
Processed 20,500,000 rows...
Processed 25,500,000 rows...
Processed 30,500,000 rows...
Processed 35,500,000 rows...


...

Processed 40,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_64547/2956955587.py:61: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 45,500,000 rows...
Processed 50,500,000 rows...
Processed 55,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_64547/2956955587.py:61: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 60,500,000 rows...
Processed 65,500,000 rows...
Processed 70,500,000 rows...

Saved to SharedResponses.parquet

Dtypes:
ResponseID                 category
UserID                       UInt64
ScenarioOrder                 UInt8
Intervention                  uint8
PedPed                        uint8
Barrier                       uint8
CrossingSignal                uint8
AttributeLevel             category
ScenarioTypeStrict         category
DefaultChoice              category
NonDefaultChoice           category
DefaultChoiceIsOmission       UInt8
NumberOfCharacters            UInt8
DiffNumberOFCharacters         Int8
Saved                         int64
UserCountry3               category
Man                           UInt8
Woman                         UInt8
Pregnant                      UInt8
Stroller                      UInt8
OldMan                        UInt8
OldWoman                      UInt8
Boy                           UInt8
Girl                          UInt8
Homele